# Model Comparison — single 70/30 split

Generates the per-window-cache features once, fixes a single 70/30 train/test split, and runs every model (sklearn baselines + custom torch heads) through the **same** train/val metric harness. Ends with one table of mean MSE and mean Pearson (PCC) across traits, for both train and validation.

All models consume the same mean-pooled, scaled feature matrix `X_proc`. Metrics are computed per-trait over non-NaN entries, then averaged across traits.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent))
from crop_embed import FixedWindowEmbedder, LinearHead, MLPHead

CACHE_PATH = Path("/home/andrew.dickson/svar/checkpoints/sativas413_1000.ckpt.pt")
PHENO_PATH = Path("/home/andrew.dickson/rice_data/RiceDiversity_44K_Phenotypes_34traits_PLINK.txt")

SEED = 45
TEST_SIZE = 0.1
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Generate data up front

Load the cache, align phenotypes to cache samples by NSFTVID, per-trait z-score (NaN-safe), mean-pool windows per sample, then scale.

In [2]:
cache_blob = torch.load(CACHE_PATH, map_location="cpu", weights_only=False)
cache_t         = cache_blob["cache"].float()            # (n_unique_windows, emb_dim)
sample_fp_index = cache_blob["sample_fp_index"].long()   # (n_samples, n_windows)
cache_samples   = cache_blob.get("sample_ids")

n_samples_cache, n_windows = sample_fp_index.shape
n_unique, emb_dim          = cache_t.shape
if cache_samples is None:
    raise ValueError("Cache file is missing 'sample_ids' — regenerate with generate_cache.py.")
print(f"cache {tuple(cache_t.shape)}  sample_fp_index {tuple(sample_fp_index.shape)}  emb_dim={emb_dim}")

cache (53005, 768)  sample_fp_index (383, 20585)  emb_dim=768


In [3]:
pheno_raw = pd.read_csv(PHENO_PATH, sep="\t")
pheno_raw["NSFTVID"] = pheno_raw["NSFTVID"].astype(int)
TRAIT_COLS = [c for c in pheno_raw.columns if c not in ("HybID", "NSFTVID")]

cache_nsftvid  = [int(s.rsplit("_", 1)[-1]) for s in cache_samples]
cache_id_to_ix = {nid: i for i, nid in enumerate(cache_nsftvid)}

pheno_matched = (
    pheno_raw[pheno_raw["NSFTVID"].isin(cache_id_to_ix)]
    .copy()
    .reset_index(drop=True)
)
cache_row_order = np.asarray(
    [cache_id_to_ix[nid] for nid in pheno_matched["NSFTVID"]], dtype=np.int64
)

Y_raw    = pheno_matched[TRAIT_COLS].values.astype(float)
Y_scaled = np.full_like(Y_raw, np.nan)
for j in range(Y_raw.shape[1]):
    col  = Y_raw[:, j]
    mask = ~np.isnan(col)
    if mask.sum() < 2:
        continue
    mu = col[mask].mean()
    sd = col[mask].std(ddof=0)
    Y_scaled[mask, j] = (col[mask] - mu) / (sd if sd > 0 else 1.0)

print(f"matched {len(pheno_matched)} / {len(pheno_raw)}   Y_scaled {Y_scaled.shape}")

matched 383 / 413   Y_scaled (383, 36)


In [4]:
# Mean-pool windows per sample, chunked so we never materialize (n_samples, n_windows, D).
fixed_embedder = FixedWindowEmbedder(cache_t, sample_fp_index)
fixed_embedder.to(device).eval()

POOL_BATCH = 8
sample_vec_t = torch.empty((n_samples_cache, emb_dim), dtype=torch.float32)
with torch.no_grad():
    for i in range(0, n_samples_cache, POOL_BATCH):
        j   = min(i + POOL_BATCH, n_samples_cache)
        idx = torch.arange(i, j, device=device)
        sample_vec_t[i:j] = fixed_embedder(idx).mean(dim=1).cpu()
fixed_embedder.to("cpu")

X_all  = sample_vec_t.numpy()[cache_row_order]
X_proc = StandardScaler().fit_transform(X_all)   # fit on all samples (shared by every model)
print(f"X_proc {X_proc.shape}")

X_proc (383, 768)


## 2. Single 70/30 split

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_proc, Y_scaled, test_size=TEST_SIZE, random_state=SEED
)
n_traits = y_train.shape[1]
print(f"train {X_train.shape[0]}  test {X_test.shape[0]}  traits {n_traits}")

train 344  test 39  traits 36


## 3. Metric harness

`trait_metrics` takes prediction/target matrices `(n_samples, n_traits)`, computes masked MSE and Pearson r per trait over observed entries, and returns the mean across traits. Same masking everywhere, so every model is scored identically.

In [6]:
def trait_metrics(pred, targ):
    n_t = targ.shape[1]
    mse = np.full(n_t, np.nan)
    pcc = np.full(n_t, np.nan)
    for j in range(n_t):
        m = ~np.isnan(targ[:, j]) & ~np.isnan(pred[:, j])
        if m.sum() < 2:
            continue
        p, t = pred[m, j], targ[m, j]
        mse[j] = np.mean((p - t) ** 2)
        if p.std() > 0 and t.std() > 0:
            pcc[j] = np.corrcoef(p, t)[0, 1]
    return {'mse_mean': float(np.nanmean(mse)), 'pcc_mean': float(np.nanmean(pcc)),
            'mse': mse, 'pcc': pcc}

## 4. Sklearn models (per-trait, NaN rows dropped)

Each estimator is cloned and fit once per trait on that trait's observed rows, then predicts the full train and test matrices. `MLP_wide` mirrors the PyTorch `MLPHead` width and is the slowest cell here (36 fits of a 5-layer net).

In [7]:
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

_mlp_kw = dict(activation="relu", solver="adam", alpha=1e-2, batch_size=32,
               learning_rate_init=1e-3, max_iter=500, early_stopping=True,
               validation_fraction=0.1, random_state=SEED)

SK_MODELS = [
    ("Ridge",      Ridge(alpha=10.0)),
    ("KNN",        KNeighborsRegressor(n_neighbors=5)),
    ("SVR_rbf",    SVR(kernel="rbf")),
    # ("MLP_narrow", MLPRegressor(hidden_layer_sizes=(emb_dim, 128, 128), **_mlp_kw)),
    # ("MLP_wide",   MLPRegressor(hidden_layer_sizes=(emb_dim, emb_dim, emb_dim, 128, 128), **_mlp_kw)),
]

def fit_predict_sklearn(base, X_tr, y_tr, X_te):
    pred_tr = np.full_like(y_tr, np.nan, dtype=float)
    pred_te = np.full((X_te.shape[0], y_tr.shape[1]), np.nan)
    for j in range(y_tr.shape[1]):
        m = ~np.isnan(y_tr[:, j])
        if m.sum() < 5:
            continue
        model = clone(base).fit(X_tr[m], y_tr[m, j])
        pred_tr[:, j] = model.predict(X_tr)
        pred_te[:, j] = model.predict(X_te)
    return pred_tr, pred_te

results = []
for name, base in SK_MODELS:
    p_tr, p_te = fit_predict_sklearn(base, X_train, y_train, X_test)
    tr_metrics = trait_metrics(p_tr, y_train)
    te_metrics = trait_metrics(p_te, y_test)
    metrics_dict = {"Model": name, "Train MSE Mean": tr_metrics['mse_mean'], "Val MSE Mean": te_metrics['mse_mean'],
                    "Train PCC Mean": tr_metrics['pcc_mean'], "Val PCC Mean": te_metrics['pcc_mean']}
    for i, trait in enumerate(TRAIT_COLS):
        metrics_dict[f'Train_{trait}_mse'] = tr_metrics['mse'][i]
        metrics_dict[f'Val_{trait}_mse'] = te_metrics['mse'][i]
        metrics_dict[f'Train_{trait}_pcc'] = tr_metrics['pcc'][i]
        metrics_dict[f'Val_{trait}_pcc'] = te_metrics['pcc'][i]

    results.append(metrics_dict)
    print(f"{name:11s} | val MSE {te_metrics['mse_mean']:.4f} | val PCC {te_metrics['pcc_mean']:.4f}")

Ridge       | val MSE 1.0707 | val PCC 0.4565
KNN         | val MSE 0.6753 | val PCC 0.5864
SVR_rbf     | val MSE 0.6221 | val PCC 0.6192


In [8]:
EVAL_TRAITS = [
    "Alkali spreading value",
    "Amylose content",
    "Panicle number per plant",
    "Protein content",
    "Seed length",
    "Seed number per panicle",
]

eval_val_cols = [f'Val_{trait}_pcc' for trait in EVAL_TRAITS]

## 5. Custom torch heads (multi-output, masked MSE)

`LinearHead` and `MLPHead` train jointly on all traits with the NaN-safe masked MSE. Inputs get a singleton window axis (`[:, None, :]`) so the head's mean-pool is a no-op on the already-pooled vectors. Train metrics are taken in eval mode (dropout off) to match how sklearn train predictions are scored.

In [23]:
from torch.utils.data import TensorDataset, DataLoader

EPOCHS = 50
LR     = 1e-3
WD     = 1e-4
N_LAYERS = 5
BATCH  = 32

def masked_mse(pred, target):
    m = ~torch.isnan(target)
    diff = (pred - torch.nan_to_num(target)) * m
    return (diff ** 2).sum(dim=0) / m.sum(dim=0).clamp(min=1)

def make_head(kind):
    if kind == "linear":
        return LinearHead(emb_dim=emb_dim, n_traits=n_traits)
    return MLPHead(emb_dim=emb_dim, n_traits=n_traits, hidden_dim=2200, n_layers=N_LAYERS, dropout=0.55)

def train_head(kind, X_tr, y_tr, X_te):
    torch.manual_seed(SEED)
    head  = make_head(kind).to(device)
    optim = torch.optim.Adam(head.parameters(), lr=LR, weight_decay=WD)

    Xtr = torch.as_tensor(X_tr, dtype=torch.float32)[:, None, :]
    ytr = torch.as_tensor(y_tr, dtype=torch.float32)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=BATCH, shuffle=True)

    head.train()
    for _ in range(EPOCHS):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optim.zero_grad()
            masked_mse(head(xb), yb).mean().backward()
            optim.step()

    head.eval()
    with torch.no_grad():
        p_tr = head(torch.as_tensor(X_tr, dtype=torch.float32)[:, None, :].to(device)).cpu().numpy()
        p_te = head(torch.as_tensor(X_te, dtype=torch.float32)[:, None, :].to(device)).cpu().numpy()
    return p_tr, p_te

for name, kind in [("Linear_head", "linear"), ("MLP_head", "mlp")]:
    p_tr, p_te = train_head(kind, X_train, y_train, X_test)
    tr_metrics = trait_metrics(p_tr, y_train)
    te_metrics = trait_metrics(p_te, y_test)
    metrics_dict = {"Model": name, "Train MSE Mean": tr_metrics['mse_mean'], "Val MSE Mean": te_metrics['mse_mean'],
                    "Train PCC Mean": tr_metrics['pcc_mean'], "Val PCC Mean": te_metrics['pcc_mean']}
    for i, trait in enumerate(TRAIT_COLS):
        metrics_dict[f'Train_{trait}_mse'] = tr_metrics['mse'][i]
        metrics_dict[f'Val_{trait}_mse'] = te_metrics['mse'][i]
        metrics_dict[f'Train_{trait}_pcc'] = tr_metrics['pcc'][i]
        metrics_dict[f'Val_{trait}_pcc'] = te_metrics['pcc'][i]
    print(f"{name:11s} | val MSE {te_metrics['mse_mean']:.4f} | val PCC {te_metrics['pcc_mean']:.4f}")
results.append(metrics_dict)

Linear_head | val MSE 0.8742 | val PCC 0.4986
MLP_head    | val MSE 0.6609 | val PCC 0.6295


## 6. Summary table

Mean MSE and mean Pearson across traits, train vs. validation, for every model. Sorted by validation PCC.

In [24]:
summary = (
    pd.DataFrame(results)
    [["Model", "Train MSE Mean", "Val MSE Mean", "Train PCC Mean", "Val PCC Mean"] + eval_val_cols]
    .sort_values("Val PCC Mean", ascending=False)
    .round(4)
    .reset_index(drop=True)
)
summary

,Model,Train MSE Mean,Val MSE Mean,Train PCC Mean,Val PCC Mean,Val_Alkali spreading value_pcc,Val_Amylose content_pcc,Val_Panicle number per plant_pcc,Val_Protein content_pcc,Val_Seed length_pcc,Val_Seed number per panicle_pcc
0,MLP_head,0.1219,0.6609,0.9669,0.6295,0.4370,0.8680,0.7839,0.3981,0.8327,0.7073
1,SVR_rbf,0.2919,0.6221,0.8744,0.6192,0.4968,0.7202,0.8302,0.3854,0.7631,0.6346
2,KNN,0.4348,0.6753,0.7524,0.5864,0.4960,0.6967,0.7996,0.4432,0.7743,0.6412
3,Ridge,0.0429,1.0707,0.9807,0.4565,0.2261,0.6101,0.6047,0.1801,0.7262,0.4661


In [21]:
summary = (
    pd.DataFrame(results)
    [["Model", "Train MSE", "Val MSE", "Train PCC", "Val PCC"]]
    .sort_values("Val PCC", ascending=False)
    .round(4)
    .reset_index(drop=True)
)
summary

KeyError: "['Train MSE', 'Val MSE', 'Train PCC', 'Val PCC'] not in index"

## 7. Add the attention head

Run `python train_attention_head.py` first — it uses this same split and metric harness and writes `comparison_results/attention_head_summary.csv`. The cell below loads that row and merges it into the table.

In [ ]:
attn = pd.read_csv("comparison_results/attention_head_summary.csv")
combined = (
    pd.concat([pd.DataFrame(results), attn], ignore_index=True)
    [["Model", "Train MSE Mean", "Val MSE Mean", "Train PCC Mean", "Val PCC Mean"] + eval_val_cols]
    .sort_values("Val PCC Mean", ascending=False)
    .round(4)
    .reset_index(drop=True)
)
combined